# 假设检验基础完整教程：从零假设到统计功效

## 📚 教学目标
1. 理解零假设和备择假设的设定规则
2. 掌握第一类错误（α）和第二类错误（β）的区别与权衡
3. 理解 p 值的含义：在 H₀ 为真时，得到至少与样本一样极端的检验统计量的概率
4. 掌握统计功效（Power = 1 - β）的概念和计算
5. 理解显著性水平的选择（0.05, 0.01）及其对结论的影响

**参考书**：《基础统计学(第14版)》（Triola）第 8-1 节
**教学策略**：先用直觉理解假设检验的逻辑，再逐步学习每一步的计算和判断

---

## 1. 场景设定：如何用数据做判断？

### 🎯 一个现实问题

XSORT 性别选择法声称可以提高生女孩的概率。在初步测试中，14 个新生儿中有 13 个是女孩。

**核心问题**：这个结果是否足以证明 XSORT 方法有效？还是仅仅是运气？

### 📖 书中的定义

> **假设检验**是一种通过样本数据对总体参数做出推断的统计方法。
> 它通过计算样本统计量与假设参数值之间的差异，来判断这种差异是否具有统计显著性。

### 💡 核心逻辑

假设检验采用"**反证法**"思维：
1. 先假设一个命题成立（零假设 H₀）
2. 在这个假设下，计算观测到当前样本（或更极端样本）的概率
3. 如果概率很小（p < α），说明"如果 H₀ 成立，这个结果几乎不可能发生"，从而拒绝 H₀
4. 如果概率不小，说明没有足够证据拒绝 H₀

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 零假设与备择假设

### 📐 假设的设定规则

假设检验涉及两个互补的假设：

**零假设 (Null Hypothesis, H₀)**：
- 包含等号（=、≤、≥）
- 代表"没有效果"、"没有差异"、"维持现状"
- 我们试图找到证据来**拒绝**它

**备择假设 (Alternative Hypothesis, H₁ 或 Hₐ)**：
- 不包含等号（≠、<、>）
- 代表研究者的命题或想要证明的结论
- 只有当拒绝 H₀ 时，我们才**接受** H₁

### 📝 设定假设的步骤

| 步骤 | 说明 |
|------|------|
| 步骤 1 | 写出原命题的数学表达式 |
| 步骤 2 | 如果原命题不成立，写出其对立面 |
| 步骤 3 | 不含等式的为 H₁，含等式的为 H₀ |

### 📖 三种检验类型

| 检验类型 | H₁ | 临界域位置 |
|---------|-----|----------|
| 双侧检验 | μ ≠ μ₀ | 两侧 |
| 左侧检验 | μ < μ₀ | 左侧 |
| 右侧检验 | μ > μ₀ | 右侧 |

In [ ]:
# ========== 假设设定示例 ==========

print("=" * 60)
print("📋 假设设定示例")
print("=" * 60)

# 示例1: XSORT 性别选择法
print("\n🎯 示例1: XSORT 性别选择法")
print("  原命题: 采用 XSORT 方法更可能生女孩 (p > 0.5)")
print("  步骤1: 原命题 p > 0.5")
print("  步骤2: 如果不成立，则 p ≤ 0.5")
print("  步骤3: p > 0.5 不含等式 → H₁")
print("  ─────────────────────────────────")
print("  H₀: p = 0.5")
print("  H₁: p > 0.5  (右侧检验)")

# 示例2: 成年人睡眠时间
print("\n🎯 示例2: 成年人睡眠时间")
print("  原命题: 平均睡眠时间少于7小时 (μ < 7)")
print("  步骤1: 原命题 μ < 7")
print("  步骤2: 如果不成立，则 μ ≥ 7")
print("  步骤3: μ < 7 不含等式 → H₁")
print("  ─────────────────────────────────")
print("  H₀: μ = 7")
print("  H₁: μ < 7  (左侧检验)")

# 示例3: 人体体温
print("\n🎯 示例3: 人体体温")
print("  原命题: 平均体温是 98.6°F (μ = 98.6)")
print("  步骤1: 原命题 μ = 98.6")
print("  步骤2: 如果不成立，则 μ ≠ 98.6")
print("  步骤3: μ ≠ 98.6 不含等式 → H₁")
print("  ─────────────────────────────────")
print("  H₀: μ = 98.6")
print("  H₁: μ ≠ 98.6  (双侧检验)")

print("\n" + "=" * 60)

---

## 3. 显著性水平与两类错误

### 📐 第一类错误与第二类错误

即使假设检验步骤都正确，结论也可能出错。我们需要区分两种错误：

**第一类错误 (Type I Error, 弃真错误)**：
- 当 H₀ 为真时，却拒绝了 H₀
- 概率 = α（显著性水平）
- 例如：药物实际无效，却得出"有效"的结论

**第二类错误 (Type II Error, 存伪错误)**：
- 当 H₀ 为假时，却没有拒绝 H₀
- 概率 = β
- 例如：药物实际有效，却得出"无效"的结论

### 📝 决策矩阵

|  | H₀ 实际为真 | H₀ 实际为假 |
|--|-----------|----------|
| **不能拒绝 H₀** | ✅ 正确决策 (1-α) | ❌ 第二类错误 (β) |
| **拒绝 H₀** | ❌ 第一类错误 (α) | ✅ 正确决策 (1-β = Power) |

### 📐 显著性水平

$$\alpha = P(\text{第一类错误}) = P(\text{拒绝 } H_0 \mid H_0 \text{ 为真})$$

常见选择：α = 0.05、0.01、0.10

In [ ]:
# ========== 两类错误的模拟 ==========

print("📊 模拟两类错误的发生频率")
print(f"  显著性水平 α = 0.05")
print(f"  模拟次数 = 10,000\n")

# 模拟参数
n_sims = 10000
alpha = 0.05
n_sample = 30
mu_0 = 100  # H₀: μ = 100
mu_true = 100  # 实际总体均值 = 100 (H₀ 为真)
sigma = 15

# 模拟当 H₀ 为真时的检验
type1_count = 0
for _ in range(n_sims):
    sample = np.random.normal(mu_true, sigma, n_sample)
    t_stat, p_val = stats.ttest_1samp(sample, mu_0)
    if p_val < alpha:
        type1_count += 1

print(f"📊 当 H₀ 为真 (μ = {mu_0}) 时:")
print(f"  错误拒绝 H₀ 的比例 (α 估计) = {type1_count/n_sims:.4f}")
print(f"  理论值 α = {alpha}")
print(f"  💡 模拟值接近理论值，验证了 α 的含义")

# 模拟当 H₀ 为假时的检验
mu_true_alt = 105  # 实际均值 = 105 (H₀ 为假)
type2_count = 0
for _ in range(n_sims):
    sample = np.random.normal(mu_true_alt, sigma, n_sample)
    t_stat, p_val = stats.ttest_1samp(sample, mu_0)
    if p_val >= alpha:
        type2_count += 1

power = 1 - type2_count / n_sims
print(f"\n📊 当 H₀ 为假 (μ = {mu_true_alt}) 时:")
print(f"  未能拒绝 H₀ 的比例 (β 估计) = {type2_count/n_sims:.4f}")
print(f"  统计功效 (Power = 1-β) = {power:.4f}")
print(f"  💡 当实际均值与假设值差距较大时，功效更高")

In [ ]:
# ========== 可视化两类错误 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 第一类错误
ax = axes[0]
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)
ax.plot(x, y, 'b-', linewidth=2, label='Sampling Dist (H₀ true)')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')

z_crit = stats.norm.ppf(1 - alpha)
x_reject = np.linspace(z_crit, 4, 500)
y_reject = stats.norm.pdf(x_reject)
ax.fill_between(x_reject, 0, y_reject, alpha=0.4, color='#e74c3c', label=f'Rejection Region (α={alpha})')
ax.axvline(x=z_crit, color='red', linestyle='--', linewidth=2, label=f'Critical Value z={z_crit:.2f}')
ax.set_xlabel('Z Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Type I Error (Reject H₀ when H₀ is True)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# 右图: 第二类错误
ax = axes[1]
# H₀ 为真的分布
x = np.linspace(-4, 8, 1000)
y_h0 = stats.norm.pdf(x, 0, 1)
ax.plot(x, y_h0, 'b-', linewidth=2, label='H₀: μ=μ₀')
ax.fill_between(x, 0, y_h0, alpha=0.1, color='steelblue')

# H₁ 为真的分布 (均值偏移)
delta = 1.5  # 效应量
y_h1 = stats.norm.pdf(x, delta, 1)
ax.plot(x, y_h1, 'g-', linewidth=2, label=f'H₁: μ=μ₀+{delta}σ')
ax.fill_between(x, 0, y_h1, alpha=0.1, color='#2ecc71')

# 标记 α 区域
x_alpha = np.linspace(z_crit, 4, 500)
ax.fill_between(x_alpha, 0, stats.norm.pdf(x_alpha, 0, 1), alpha=0.4, color='#e74c3c', label='α (Type I)')

# 标记 β 区域
x_beta = np.linspace(-4, z_crit, 500)
ax.fill_between(x_beta, 0, stats.norm.pdf(x_beta, delta, 1), alpha=0.4, color='#e67e22', label='β (Type II)')

ax.axvline(x=z_crit, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.set_xlabel('Z Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Type I vs Type II Errors', fontsize=14, fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：当 H₀ 为真时，红色区域（α）是错误拒绝 H₀ 的概率")
print("  右图：α（红色）和 β（橙色）的关系——减小 α 会增大 β，反之亦然")
print("  统计功效 = 1 - β = 绿色分布中右侧的面积")

---

## 4. p 值的含义与计算

### 📐 p 值的定义

**p 值**是在假设原假设 H₀ 为真的前提下，得到至少与实际样本的检验统计量一样极端的统计量的概率。

$$p\text{值} = P(\text{检验统计量至少与样本一样极端} \mid H_0 \text{ 为真})$$

### 📝 p 值的计算方法

| 检验类型 | p 值计算 |
|---------|----------|
| 左侧检验 (H₁: μ < μ₀) | p = 检验统计量左侧的面积 |
| 右侧检验 (H₁: μ > μ₀) | p = 检验统计量右侧的面积 |
| 双侧检验 (H₁: μ ≠ μ₀) | p = 2 × 超出 |检验统计量| 的面积 |

### 📐 判断规则

$$\text{如果 } p\text{值} < \alpha \text{，则拒绝 } H_0$$
$$\text{如果 } p\text{值} \geq \alpha \text{，则不能拒绝 } H_0$$

### ⚠️ 注意区分

- p 值 ≠ H₀ 为真的概率
- p 值 ≠ "犯错误的概率"
- p 值 = 在 H₀ 为真的假设下，观测到当前或更极端结果的概率

In [ ]:
# ========== p 值计算示例 ==========

print("=" * 60)
print("📋 p 值计算示例：三种检验类型")
print("=" * 60)

# 假设检验统计量 z = 1.85
z = 1.85

# 右侧检验: p = P(Z > z)
p_right = 1 - stats.norm.cdf(z)
print(f"\n📊 右侧检验 (H₁: μ > μ₀)")
print(f"  检验统计量 z = {z}")
print(f"  p 值 = P(Z > {z}) = {p_right:.4f}")
print(f"  即 z 值右侧的面积")

# 左侧检验: p = P(Z < z)
p_left = stats.norm.cdf(z)
print(f"\n📊 左侧检验 (H₁: μ < μ₀)")
print(f"  检验统计量 z = {z}")
print(f"  p 值 = P(Z < {z}) = {p_left:.4f}")
print(f"  即 z 值左侧的面积")

# 双侧检验: p = 2 * min(左侧, 右侧)
p_two = 2 * min(p_left, p_right)
print(f"\n📊 双侧检验 (H₁: μ ≠ μ₀)")
print(f"  检验统计量 z = {z}")
print(f"  p 值 = 2 × min({p_left:.4f}, {p_right:.4f}) = {p_two:.4f}")
print(f"  即两侧超出 |z| 的面积之和")

# 判断
alpha = 0.05
print(f"\n{'='*60}")
print(f"📋 判断 (α = {alpha})")
print(f"  右侧检验: p = {p_right:.4f} {'< α → 拒绝 H₀' if p_right < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  左侧检验: p = {p_left:.4f} {'< α → 拒绝 H₀' if p_left < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  双侧检验: p = {p_two:.4f} {'< α → 拒绝 H₀' if p_two < alpha else '≥ α → 不能拒绝 H₀'}")

In [ ]:
# ========== 可视化三种检验类型的 p 值 ==========

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
z_val = 1.85

# 右侧检验
ax = axes[0]
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)
ax.plot(x, y, 'b-', linewidth=2)
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')
x_p = np.linspace(z_val, 4, 500)
ax.fill_between(x_p, 0, stats.norm.pdf(x_p), alpha=0.4, color='#e74c3c', label=f'p = {1-stats.norm.cdf(z_val):.4f}')
ax.axvline(x=z_val, color='red', linestyle='--', linewidth=2)
ax.set_title('Right-Tailed Test', fontsize=14, fontweight='bold')
ax.set_xlabel('Z Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# 左侧检验
ax = axes[1]
ax.plot(x, y, 'b-', linewidth=2)
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')
x_p = np.linspace(-4, z_val, 500)
ax.fill_between(x_p, 0, stats.norm.pdf(x_p), alpha=0.4, color='#e74c3c', label=f'p = {stats.norm.cdf(z_val):.4f}')
ax.axvline(x=z_val, color='red', linestyle='--', linewidth=2)
ax.set_title('Left-Tailed Test', fontsize=14, fontweight='bold')
ax.set_xlabel('Z Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# 双侧检验
ax = axes[2]
ax.plot(x, y, 'b-', linewidth=2)
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')
x_p = np.linspace(z_val, 4, 500)
ax.fill_between(x_p, 0, stats.norm.pdf(x_p), alpha=0.4, color='#e74c3c')
x_p = np.linspace(-4, -z_val, 500)
ax.fill_between(x_p, 0, stats.norm.pdf(x_p), alpha=0.4, color='#e74c3c', label=f'p = {2*(1-stats.norm.cdf(z_val)):.4f}')
ax.axvline(x=z_val, color='red', linestyle='--', linewidth=2)
ax.axvline(x=-z_val, color='red', linestyle='--', linewidth=2)
ax.set_title('Two-Tailed Test', fontsize=14, fontweight='bold')
ax.set_xlabel('Z Value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"💡 图解说明：")
print(f"  红色区域 = p 值对应的概率面积")
print(f"  检验统计量 z = {z_val}")
print(f"  右侧检验 p 值 = 右侧尾部面积")
print(f"  左侧检验 p 值 = 左侧尾部面积")
print(f"  双侧检验 p 值 = 两侧尾部面积之和")

---

## 5. 检验统计量与临界值法

### 📐 检验统计量

检验统计量是将样本统计量转换为标准分数，用于判断样本与假设之间的差异。

**比例检验统计量**（本章重点）：

$$z = \frac{\hat{p} - p}{\sqrt{\frac{pq}{n}}}$$

其中：
- $\hat{p}$ = 样本比例
- p = 原假设中的总体比例
- q = 1 - p
- n = 样本量

**均值检验统计量**（σ 已知时）：

$$z = \frac{\bar{x} - \mu}{\sigma / \sqrt{n}}$$

**均值检验统计量**（σ 未知时）：

$$t = \frac{\bar{x} - \mu}{s / \sqrt{n}}, \quad df = n - 1$$

### 📐 临界值法

临界值是分开拒绝域和非拒绝域的数值。

| 检验类型 | 临界值 |
|---------|--------|
| 右侧检验 | $z_{\alpha}$ (右侧面积 = α) |
| 左侧检验 | $-z_{\alpha}$ (左侧面积 = α) |
| 双侧检验 | $\pm z_{\alpha/2}$ (每侧面积 = α/2) |

**判断规则**：检验统计量落在拒绝域内 → 拒绝 H₀

In [ ]:
# ========== 临界值法示例 ==========

print("=" * 60)
print("📋 临界值法示例")
print("=" * 60)

alpha = 0.05

# 右侧检验临界值
z_crit_right = stats.norm.ppf(1 - alpha)
print(f"\n📊 右侧检验 (α = {alpha})")
print(f"  临界值 z_α = {z_crit_right:.3f}")
print(f"  拒绝域: z > {z_crit_right:.3f}")
print(f"  判断: 如果检验统计量 > {z_crit_right:.3f}，则拒绝 H₀")

# 左侧检验临界值
z_crit_left = stats.norm.ppf(alpha)
print(f"\n📊 左侧检验 (α = {alpha})")
print(f"  临界值 -z_α = {z_crit_left:.3f}")
print(f"  拒绝域: z < {z_crit_left:.3f}")
print(f"  判断: 如果检验统计量 < {z_crit_left:.3f}，则拒绝 H₀")

# 双侧检验临界值
z_crit_two = stats.norm.ppf(1 - alpha/2)
print(f"\n📊 双侧检验 (α = {alpha})")
print(f"  临界值 ±z_{{α/2}} = ±{z_crit_two:.3f}")
print(f"  拒绝域: z < -{z_crit_two:.3f} 或 z > {z_crit_two:.3f}")
print(f"  判断: 如果 |检验统计量| > {z_crit_two:.3f}，则拒绝 H₀")

---

## 6. 完整示例：双重认证使用比例检验

### 🎯 问题描述

在一项调查中，926 位互联网用户中有 52% 使用双重认证保护数据。

检验命题："大多数互联网用户使用双重认证来保护他们的网络数据"（p > 0.5）

### 📐 假设设定

$H_0: p = 0.5$
$H_1: p > 0.5$ （原命题，右侧检验）

### 📐 检验统计量

$$z = \frac{\hat{p} - p}{\sqrt{\frac{pq}{n}}} = \frac{0.52 - 0.5}{\sqrt{\frac{(0.5)(0.5)}{926}}} = 1.25$$

In [ ]:
# ========== 双重认证使用比例检验 ==========

print("=" * 60)
print("📋 双重认证使用比例 - 完整假设检验")
print("=" * 60)

# 数据
n = 926
p_hat = 0.52
p_0 = 0.50  # H₀ 中的比例
alpha = 0.05

print(f"\n📊 步骤 1-3: 假设设定")
print(f"  H₀: p = {p_0}")
print(f"  H₁: p > {p_0} (原命题)")
print(f"  显著性水平 α = {alpha}")

# 检查条件
print(f"\n📊 步骤 4-5: 检查条件")
print(f"  样本量 n = {n}")
print(f"  np₀ = {n * p_0:.0f} ≥ 5 ✓")
print(f"  nq₀ = {n * (1-p_0):.0f} ≥ 5 ✓")
print(f"  条件满足，可以使用正态近似")

# 计算检验统计量
z_stat = (p_hat - p_0) / np.sqrt(p_0 * (1 - p_0) / n)
print(f"\n📊 步骤 6: 计算检验统计量")
print(f"  z = (p̂ - p) / √(pq/n)")
print(f"  z = ({p_hat} - {p_0}) / √({p_0}×{1-p_0}/{n})")
print(f"  z = {z_stat:.4f}")

# 计算 p 值 (右侧检验)
p_value = 1 - stats.norm.cdf(z_stat)
print(f"\n📊 步骤 6 (续): 计算 p 值")
print(f"  右侧检验: p 值 = P(Z > {z_stat:.2f})")
print(f"  p 值 = {p_value:.4f}")

# 临界值
z_critical = stats.norm.ppf(1 - alpha)
print(f"\n📊 步骤 6 (续): 临界值法")
print(f"  临界值 z_α = {z_critical:.3f}")
print(f"  检验统计量 z = {z_stat:.4f}")

# 判断
print(f"\n📊 步骤 7: 做出判断")
print(f"  p 值法: p = {p_value:.4f} {'< α → 拒绝 H₀' if p_value < alpha else '≥ α → 不能拒绝 H₀'}")
print(f"  临界值法: z = {z_stat:.4f} {'> z_α → 拒绝 H₀' if z_stat > z_critical else '≤ z_α → 不能拒绝 H₀'}")

# 结论
print(f"\n📊 步骤 8: 结论")
if p_value >= alpha:
    print(f"  没有足够的样本证据支持'大多数互联网用户使用双重认证'")
else:
    print(f"  有足够的证据支持'大多数互联网用户使用双重认证'")

print("\n" + "=" * 60)

In [ ]:
# ========== 可视化检验结果 ==========

fig, ax = plt.subplots(figsize=(10, 6))

x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)
ax.plot(x, y, 'b-', linewidth=2, label='Standard Normal')
ax.fill_between(x, 0, y, alpha=0.1, color='steelblue')

# 标记拒绝域
x_reject = np.linspace(z_critical, 4, 500)
y_reject = stats.norm.pdf(x_reject)
ax.fill_between(x_reject, 0, y_reject, alpha=0.4, color='#e74c3c', label=f'Rejection Region (α={alpha})')

# 标记临界值
ax.axvline(x=z_critical, color='red', linestyle='--', linewidth=2, label=f'Critical Value z={z_critical:.3f}')

# 标记检验统计量
ax.axvline(x=z_stat, color='#e67e22', linestyle='-', linewidth=2.5, label=f'Test Statistic z={z_stat:.2f}')

ax.set_xlabel('Z Value', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Hypothesis Test: Right-Tailed (p > 0.5)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"💡 图解说明：")
print(f"  蓝色曲线：标准正态分布")
print(f"  红色区域：拒绝域 (α = {alpha})")
print(f"  红色虚线：临界值 z = {z_critical:.3f}")
print(f"  橙色实线：检验统计量 z = {z_stat:.2f}")
print(f"  检验统计量未落在拒绝域内 → 不能拒绝 H₀")

---

## 7. 统计功效 (Power)

### 📐 统计功效的定义

**统计功效**是当 H₀ 为假时，正确拒绝 H₀ 的概率：

$$\text{Power} = 1 - \beta = P(\text{拒绝 } H_0 \mid H_1 \text{ 为真})$$

### 📝 影响功效的因素

| 因素 | 增大时对功效的影响 |
|------|------------------|
| 样本量 n | ↑ 功效增大 |
| 显著性水平 α | ↑ 功效增大 |
| 效应量 (真实值与假设值的差距) | ↑ 功效增大 |

### 📖 书中的观点

> 当总体比例 p 的真实值为 0.6 时，有 18.0% 的概率可以正确拒绝 H₀: p = 0.5。
> 当真实值为 0.9 时，有 98.8% 的概率可以正确拒绝。
> 参数真实值与假设值的差值越大，统计功效越大。

In [ ]:
# ========== 统计功效计算 ==========

print("=" * 60)
print("📋 统计功效计算：XSORT 例子")
print("=" * 60)

# 参数
n = 14  # 书中例3: 14 个新生儿
p_0 = 0.5  # H₀: p = 0.5
alpha = 0.05

# 不同的真实比例
true_p_values = [0.6, 0.7, 0.8, 0.9]

print(f"\n📊 参数设置:")
print(f"  n = {n}")
print(f"  H₀: p = {p_0}")
print(f"  H₁: p > {p_0}")
print(f"  α = {alpha}")

# 计算临界值 (右侧检验)
z_alpha = stats.norm.ppf(1 - alpha)
# 临界比例: p_crit = p_0 + z_alpha * sqrt(p_0*q_0/n)
p_crit = p_0 + z_alpha * np.sqrt(p_0 * (1 - p_0) / n)
print(f"  临界值 z_α = {z_alpha:.3f}")
print(f"  临界比例 p_crit = {p_crit:.4f}")

print(f"\n{'真实比例 p':<15} {'β (Type II)':<15} {'Power (1-β)':<15}")
print("-" * 45)

for p_true in true_p_values:
    # 计算在真实比例下，不能拒绝 H₀ 的概率 (β)
    # β = P(p̂ < p_crit | p = p_true)
    # z = (p_crit - p_true) / sqrt(p_true*q_true/n)
    se_true = np.sqrt(p_true * (1 - p_true) / n)
    z_beta = (p_crit - p_true) / se_true
    beta = stats.norm.cdf(z_beta)
    power = 1 - beta
    print(f"  {p_true:<15.1f} {beta:<15.3f} {power:<15.3f}")

print(f"\n💡 解读:")
print(f"  当真实比例 p = 0.6 时，只有约 18% 的概率能正确拒绝 H₀")
print(f"  当真实比例 p = 0.9 时，有约 99% 的概率能正确拒绝 H₀")
print(f"  效应量越大（真实值离假设值越远），统计功效越高")
print(f"  通常要求 Power ≥ 0.80 作为实验设计的标准")

In [ ]:
# ========== 功效曲线可视化 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图: 功效与真实比例的关系
ax = axes[0]
n = 14
p_0 = 0.5
alpha = 0.05
z_alpha = stats.norm.ppf(1 - alpha)
p_crit = p_0 + z_alpha * np.sqrt(p_0 * (1 - p_0) / n)

p_true_range = np.linspace(0.51, 0.95, 100)
powers = []
for p_true in p_true_range:
    se_true = np.sqrt(p_true * (1 - p_true) / n)
    z_beta = (p_crit - p_true) / se_true
    beta = stats.norm.cdf(z_beta)
    powers.append(1 - beta)

ax.plot(p_true_range, powers, 'b-', linewidth=2)
ax.axhline(y=0.80, color='green', linestyle='--', alpha=0.7, label='Power = 0.80 (Standard)')
ax.axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='α = 0.05')
ax.fill_between(p_true_range, 0.80, powers, where=[p >= 0.80 for p in powers], alpha=0.2, color='green')
ax.set_xlabel('True Proportion p', fontsize=12)
ax.set_ylabel('Statistical Power', fontsize=12)
ax.set_title(f'Power vs True Proportion (n={n}, α={alpha})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

# 右图: 功效与样本量的关系
ax = axes[1]
p_true = 0.7
n_range = np.arange(10, 200, 5)
powers_n = []
for n_val in n_range:
    p_crit_n = p_0 + z_alpha * np.sqrt(p_0 * (1 - p_0) / n_val)
    se_true_n = np.sqrt(p_true * (1 - p_true) / n_val)
    z_beta_n = (p_crit_n - p_true) / se_true_n
    beta_n = stats.norm.cdf(z_beta_n)
    powers_n.append(1 - beta_n)

ax.plot(n_range, powers_n, 'b-', linewidth=2)
ax.axhline(y=0.80, color='green', linestyle='--', alpha=0.7, label='Power = 0.80')
ax.set_xlabel('Sample Size n', fontsize=12)
ax.set_ylabel('Statistical Power', fontsize=12)
ax.set_title(f'Power vs Sample Size (p_true={p_true}, α={alpha})', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("💡 图解说明：")
print("  左图：固定样本量 n=14，功效随真实比例增大而增大")
print("  右图：固定真实比例 p=0.7，功效随样本量增大而增大")
print("  绿色虚线：Power = 0.80 的标准线")
print("  要达到 80% 功效，需要足够大的样本量或效应量")

---

## 8. 完整假设检验流程总结

### 📝 八步流程 (图 8-1)

假设检验的标准流程如下：

| 步骤 | 内容 |
|------|------|
| 步骤 1 | 写出原命题的数学表达式 |
| 步骤 2 | 如果原命题不成立，写出其对立面 |
| 步骤 3 | 设定 H₀ 和 H₁ |
| 步骤 4 | 选择显著性水平 α |
| 步骤 5 | 选择检验统计量（z、t、χ² 等） |
| 步骤 6 | 计算检验统计量，求 p 值或临界值 |
| 步骤 7 | 做出拒绝或不能拒绝 H₀ 的判断 |
| 步骤 8 | 用非技术语言写出结论 |

### 📖 结论模板 (图 8-5)

| 原命题 | 判断 | 结论 |
|--------|------|------|
| 不含等式 | 拒绝 H₀ | 有足够的证据支持以下说法：（原命题） |
| 不含等式 | 不能拒绝 H₀ | 没有足够的证据支持以下说法：（原命题） |
| 含等式 | 拒绝 H₀ | 有足够的证据可以拒绝：（原命题） |
| 含等式 | 不能拒绝 H₀ | 没有足够的证据可以拒绝：（原命题） |

### ⚠️ 重要提醒

- **不要说"接受 H₀"**，应说"不能拒绝 H₀"
- **用非技术语言总结**：让没有统计知识的人也能理解
- **避免多重否定**：简化表达，确保结论清晰

In [ ]:
# ========== 完整流程示例 ==========

print("=" * 60)
print("📋 完整假设检验流程示例")
print("=" * 60)

# 数据: 成年人睡眠时间
sleep_data = np.array([4.8, 4.4, 8.6, 6.9, 7.7, 1.0, 7.0, 8.0, 6.5, 7.2, 5.5, 8.1])
n = len(sleep_data)
x_bar = np.mean(sleep_data)
s = np.std(sleep_data, ddof=1)
mu_0 = 7
alpha = 0.05

print(f"\n🎯 命题: 成年人平均睡眠时间少于7小时")

# 步骤 1-3: 假设
print(f"\n📊 步骤 1-3: 假设设定")
print(f"  H₀: μ = {mu_0}")
print(f"  H₁: μ < {mu_0} (原命题)")

# 步骤 4: 显著性水平
print(f"\n📊 步骤 4: 显著性水平 α = {alpha}")

# 步骤 5: 检验统计量选择
print(f"\n📊 步骤 5: 选择 t 检验 (σ 未知)")
print(f"  样本量 n = {n}")
print(f"  样本均值 x̄ = {x_bar:.4f}")
print(f"  样本标准差 s = {s:.4f}")
print(f"  自由度 df = {n-1}")

# 步骤 6: 计算
t_stat = (x_bar - mu_0) / (s / np.sqrt(n))
p_value = stats.t.cdf(t_stat, df=n-1)  # 左侧检验

print(f"\n📊 步骤 6: 计算检验统计量")
print(f"  t = (x̄ - μ₀) / (s/√n)")
print(f"  t = ({x_bar:.4f} - {mu_0}) / ({s:.4f}/√{n})")
print(f"  t = {t_stat:.4f}")
print(f"  p 值 = P(T < {t_stat:.4f}) = {p_value:.4f}")

# 步骤 7: 判断
print(f"\n📊 步骤 7: 判断")
if p_value < alpha:
    print(f"  p = {p_value:.4f} < α = {alpha} → 拒绝 H₀")
else:
    print(f"  p = {p_value:.4f} ≥ α = {alpha} → 不能拒绝 H₀")

# 步骤 8: 结论
print(f"\n📊 步骤 8: 结论")
if p_value < alpha:
    print(f"  有足够的证据支持'成年人平均睡眠时间少于7小时'")
else:
    print(f"  没有足够的证据支持'成年人平均睡眠时间少于7小时'")

# scipy 验证
t_scipy, p_scipy = stats.ttest_1samp(sleep_data, mu_0)
print(f"\n🔬 scipy.stats.ttest_1samp 验证:")
print(f"  手算 t = {t_stat:.6f}")
print(f"  scipy t = {t_scipy:.6f}")
print(f"  手算 p = {p_value:.6f}")
print(f"  scipy p = {p_scipy:.6f} (双侧)")
print(f"  单侧 p = {p_scipy/2:.6f}")
print(f"\n  ✅ 验证通过！")

print("\n" + "=" * 60)

---

## 📌 核心概念回顾

### 📌 零假设 (H₀) 与备择假设 (H₁)
- **定义**: H₀ 含等号（=、≤、≥），H₁ 不含等号（≠、<、>）
- **设定规则**: 不含等式的原命题 → H₁；含等式的 → H₀
- **含义**: 我们试图找到证据拒绝 H₀，接受 H₁

### 📌 第一类错误与第二类错误
- **第一类错误 (α)**: H₀ 为真时拒绝 H₀（弃真错误）
- **第二类错误 (β)**: H₀ 为假时未拒绝 H₀（存伪错误）
- **关系**: α 和 β 此消彼长，增大样本量可同时减小两者

### 📌 p 值
- **定义**: 在 H₀ 为真时，得到至少与样本一样极端的检验统计量的概率
- **判断**: p < α → 拒绝 H₀；p ≥ α → 不能拒绝 H₀
- **注意**: p 值 ≠ H₀ 为真的概率

### 📌 统计功效 (Power)
- **定义**: Power = 1 - β = P(拒绝 H₀ | H₁ 为真)
- **影响因素**: 样本量↑、α↑、效应量↑ → Power↑
- **标准**: Power ≥ 0.80 被认为是可接受的

### 🔗 完整流程
```
设定假设 → 选择α → 选检验统计量 → 计算统计量 → 求p值/临界值 → 判断 → 结论
    ↓          ↓          ↓              ↓              ↓           ↓       ↓
  H₀/H₁     0.05      z/t/χ²        代入公式       查表或软件   p<α?   非技术语言
```

### 📝 检验方法汇总

| 检验类型 | 检验统计量 | 分布 | 条件 |
|---------|-----------|------|------|
| 比例 z 检验 | z = (p̂-p)/√(pq/n) | 正态 | np≥5, nq≥5 |
| 均值 t 检验 | t = (x̄-μ)/(s/√n) | t, df=n-1 | 正态或 n>30 |
| 方差 χ² 检验 | χ² = (n-1)s²/σ₀² | χ², df=n-1 | 正态分布 |

---

## ❌ 常见误区

### ❌ 误区 1: "不能拒绝 H₀" 等于 "接受 H₀"
**✓ 正确理解**: 我们永远不能"证明"或"接受"原假设。"不能拒绝"仅表示现有证据不足以推翻 H₀，但不意味着 H₀ 为真。就像法庭判决"无罪"不等于"清白"，只是"证据不足"。

### ❌ 误区 2: p 值是 H₀ 为真的概率
**✓ 正确理解**: p 值是在假设 H₀ 为真的前提下，观测到当前或更极端结果的概率。p 值 ≠ P(H₀ 为真)。p 值描述的是数据与假设的兼容性，而非假设本身的真假概率。

### ❌ 误区 3: 统计显著等于实际显著
**✓ 正确理解**: 统计显著只意味着差异不太可能由随机误差引起，但不代表差异有实际意义。例如，某安眠药让人多睡 6 分钟，统计上显著但实际意义有限。

### ❌ 误区 4: 样本量越大越好
**✓ 正确理解**: 增大样本量可以提高统计功效，但也可能使微不足道的差异变得"统计显著"。关键是要根据效应量和实际需求确定合适的样本量。

### ❌ 误区 5: α = 0.05 是唯一正确的选择
**✓ 正确理解**: 0.05 是一个约定俗成的选择，但不是唯一标准。在某些领域（如粒子物理），需要 α = 0.005 或更低。应根据具体情境和错误后果来选择合适的显著性水平。